# Ejercicio: Triggers

Antes de ejecutar, abre una terminal y deja corriendo:

```
nc -lk 9999
```

El trigger se dispara cada minuto. Los resultados del sink `console` aparecen en la terminal desde donde lanzaste Jupyter.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

spark = SparkSession.builder \
    .appName("Streaming IABD WordCount") \
    .master("local[2]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
lineasDF = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", "9999") \
    .load()

palabrasDF = lineasDF.select(explode(split(lineasDF.value, " ")).alias("palabra"))
cantidadDF = palabrasDF.groupBy("palabra").count()

## Consulta con trigger de 1 minuto

In [ ]:
q = cantidadDF.writeStream \
    .format("console") \
    .outputMode("complete") \
    .trigger(processingTime="1 minute") \
    .start()

Envía varias frases por `nc` con menos de un minuto de separación entre ellas y observa cuándo aparece el resultado en consola.

Cuando termines, detén la consulta:

In [ ]:
q.stop()

## Conclusión

Con `processingTime="1 minute"` Spark acumula los datos que llegan durante el intervalo y los procesa juntos al cumplirse el minuto. El resultado no aparece al enviar cada frase, sino cuando se dispara el trigger. Sin trigger, cada micro-batch se ejecuta apenas llegan los datos.

In [ ]:
spark.stop()